# Stage 1 · 任务学习
复用已训练 OpenJev-4B，BF16 骨干 + LoRA，FP32 原决策头与交互层。先实测单卡 pilot，再开始正式训练。

In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/SA-Jev')
    PROJECT = Path('/content/graphrag-jev')
except ImportError:
    DRIVE = Path(os.environ.get('SAJEV_WORKDIR', '../outputs/colab_local')).resolve()
    PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DRIVE.mkdir(parents=True, exist_ok=True)
REPO_URL = os.environ.get('SAJEV_REPO_URL', 'https://github.com/BoldHu/SA-Jev.git')  # Override with SAJEV_REPO_URL if needed.
SYNC_GIT = True
if not PROJECT.exists():
    if REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
    elif (DRIVE / 'sa-jev-source.zip').exists():
        PROJECT.mkdir(parents=True)
        with zipfile.ZipFile(DRIVE / 'sa-jev-source.zip') as archive:
            archive.extractall(PROJECT)
    else:
        raise RuntimeError('Set SAJEV_REPO_URL or copy sa-jev-source.zip to MyDrive/SA-Jev/')
if SYNC_GIT and (PROJECT / '.git').exists():
    status = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True)
    if status.strip():
        print('Local edits present; preserving them. Git synchronization skipped.')
    else:
        subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
os.chdir(PROJECT)
MODE = 'full'  # Reuse the prepared data; RUN_FULL_STAGE1 below controls training size.
ROOT = DRIVE / 'public_lightrag_v1' / MODE
ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(DRIVE / 'huggingface')
PUBLIC_ASSETS = DRIVE / 'public_lightrag_assets'
CONFIG_PATH = PROJECT / 'configs/public_lightrag.yaml'
print('Code:', PROJECT, 'Artifacts:', ROOT)


In [ ]:
# Run this in each fresh Colab runtime. No model weights are downloaded here.
subprocess.run([sys.executable, 'scripts/bootstrap_upstream.py'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[train,retrieval,test]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', 'vendor/OpenJev'], check=True)
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'fla-core==0.5.2'], check=True)
# Activate editable-install .pth files in this already-running notebook kernel.
import importlib, site
site_dirs = site.getsitepackages()
if site.ENABLE_USER_SITE:
    site_dirs.append(site.getusersitepackages())
for site_dir in dict.fromkeys(site_dirs):
    site.addsitedir(site_dir)
importlib.invalidate_caches()
import sa_jev
print('SA-Jev source:', sa_jev.__file__)
# If package installation asks for a runtime restart, restart before importing torch.
from sa_jev.experiments import load_config, environment_manifest
config = load_config(CONFIG_PATH)
environment_manifest(ROOT / 'environment.json', config)
def cli(command, *args):
    from collections import deque
    tail = deque(maxlen=40)
    cmd = [sys.executable, '-u', '-m', 'sa_jev.cli', command, '--root', str(ROOT), '--config', str(CONFIG_PATH), *map(str,args)]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                tail.append(line)
            returncode = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if returncode:
        raise RuntimeError(f'sajev {command} failed (exit {returncode}). Last output: ' + ''.join(tail))


In [ ]:
# Read the same credential file in every notebook. Fill it in once.
from sa_jev.environment import load_environment
ENV_FILE = (DRIVE if 'google.colab' in sys.modules else PROJECT) / '.env'
credential_status = load_environment(ENV_FILE, template=PROJECT / '.env.example')
print('Credential file:', ENV_FILE)
for name in ('OPENAI_API_KEY', 'OPENAI_BASE_URL', 'TYPESAFE_API_KEY', 'HF_TOKEN'):
    print(name + ':', 'loaded' if credential_status[name] else 'empty')
if not credential_status['OPENAI_API_KEY']:
    print('Fill OPENAI_API_KEY in the file above, save it, then rerun this cell before building the index or answering questions.')


In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime'
print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory/1024**3, 'GiB')
subprocess.run([sys.executable, 'scripts/check_qwen_lora.py', '--device', 'cuda', '--output', str(ROOT/'qwen_lora_check.json')], check=True)

In [ ]:
# Reuse full prepared data. This switch controls training, not the data directory.
import copy, importlib, torch
from sa_jev import training, experiments
for module in (training, experiments):
    importlib.reload(module)
RUN_FULL_STAGE1 = False
PROBE_STEPS = 20
TRAIN_SEEDS = [17]  # After seed 17 completes, run [29, 43] for the remaining formal seeds.
assert torch.cuda.is_available(), 'Select an A100 GPU runtime'
GPU_GIB = torch.cuda.get_device_properties(0).total_memory / 1024**3
config['model']['branch_microbatch'] = 1
config['training']['memory_limit_gib'] = min(38.0, GPU_GIB - 2.0)
# Accumulation keeps one task state's graph at a time; retain the effective batch.
config['training']['accumulation'] = 8
print('Data ROOT:', ROOT, flush=True)
print('GPU:', torch.cuda.get_device_name(0), 'capacity GiB:', round(GPU_GIB, 2), flush=True)
print('Native BF16:', torch.cuda.is_bf16_supported(including_emulation=False), flush=True)
print('Memory stop threshold GiB:', config['training']['memory_limit_gib'], flush=True)
print('Mode:', 'formal Stage 1' if RUN_FULL_STAGE1 else '20-step probe in a separate directory', flush=True)


In [ ]:
from datetime import datetime, timezone
from uuid import uuid4
from sa_jev.experiments import train_variant
stage1_config = copy.deepcopy(config)
if not RUN_FULL_STAGE1:
    stage1_config['training']['validate_every'] = 10
    stage1_config['training']['save_every'] = 10
probe_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid4().hex[:6]
for seed in (TRAIN_SEEDS if RUN_FULL_STAGE1 else [17]):
    probe_output = None if RUN_FULL_STAGE1 else ROOT / 'runs' / 'stage1_probe' / probe_id / f'seed_{seed}'
    history = train_variant(
        ROOT, stage1_config, 'stage1', seed,
        max_steps=None if RUN_FULL_STAGE1 else PROBE_STEPS,
        validation_limit=None if RUN_FULL_STAGE1 else 6,
        output_directory=probe_output,
    )
    print('Finished steps:', len(history),
          'peak reserved GiB:', max(row['peak_reserved_gib'] for row in history),
          'total valid rank observations:', sum(row['valid_rank'] for row in history), flush=True)
    if probe_output is not None:
        print('Probe saved separately:', probe_output, flush=True)


检查 training.csv/PNG/PDF、验证 loss、实际显存和 ETA。4GB 本地 tiny 测试不能代替 4B A100 测试；OOM 时降低 branch_microbatch，不切分 softmax，不截断证据，不改变标签。恢复训练使用 train_variant(..., resume=checkpoint_path)。